# Node Embeddings: Introduction

## Learning Objectives

1. **Explain** why graph data needs specialized embedding methods
2. **Define** the encoder-decoder framework for graph embeddings
3. **List** desirable properties of a node embedding
4. **Preview** the DeepWalk, Node2Vec, and GNN approaches

## Why Embed Graphs?

Many ML algorithms expect fixed-size vector inputs. Graphs are irregular:
- Variable number of neighbors
- No natural ordering of nodes
- Different graph sizes

**Goal:** map each node $v$ to a vector $z_v \in \mathbb{R}^d$ such that:
$$\text{sim}_{\text{graph}}(u, v) \approx z_u \cdot z_v$$

Where "similar in graph" captures some notion of proximity or structural role.

## The Encoder-Decoder Framework

**Encoder:** maps node $v$ to embedding $z_v = ENC(v)$

**Decoder:** reconstructs pairwise similarity from embeddings:
$$\hat{S}_{uv} = DEC(z_u, z_v) = \sigma(z_u^\top z_v)$$

**Training objective:** minimize reconstruction loss over observed graph structure:
$$\mathcal{L} = \sum_{(u,v)} \ell(S_{uv}, DEC(z_u, z_v))$$

Different algorithms differ in:
1. What $S_{uv}$ means (adjacency? random walk co-occurrence?)
2. How the encoder $ENC$ works (lookup? neural network?)

## What Should Embeddings Capture?

| Property | Description |
|----------|-------------|
| **Local neighborhood** | Nodes with many common neighbors should be close |
| **Global structure** | Nodes in the same community should be close |
| **Structural roles** | Nodes with similar roles (e.g., hubs) should be close |
| **Feature alignment** | Node features should be consistent with embedding |

**Different algorithms make different trade-offs:**
- DeepWalk/Node2Vec: optimize for short-range random walk co-occurrence
- Spectral embeddings: optimize for global graph structure (Laplacian eigenvectors)
- GNNs: use node features + graph structure together

In [1]:
from collections import defaultdict
# Demonstrate the basic encoder-decoder idea
import math, random

def sigmoid(x): return 1.0/(1.0+math.exp(-x))
def dot(a, b): return sum(x*y for x,y in zip(a,b))

# Simple graph: adjacency list
edges = [(0,1),(0,2),(1,2),(3,4),(3,5),(4,5),(2,3)]
n_nodes = 6
adj = defaultdict(set)
for u,v in edges:
    adj[u].add(v); adj[v].add(u)

# Random node embeddings (dim=2)
rng = random.Random(7)
d = 2
Z = [[rng.gauss(0,0.1) for _ in range(d)] for _ in range(n_nodes)]

# Pairwise similarities: predicted and actual
print("Node pair | Graph adjacent | Sigmoid(z_u · z_v)")
print("-"*52)
for u in range(n_nodes):
    for v in range(u+1, n_nodes):
        actual = 1 if v in adj[u] else 0
        pred = sigmoid(dot(Z[u], Z[v]))
        print(f"  ({u},{v})     |       {actual}          |  {pred:.4f}")

Node pair | Graph adjacent | Sigmoid(z_u · z_v)
----------------------------------------------------
  (0,1)     |       1          |  0.4997
  (0,2)     |       1          |  0.5003
  (0,3)     |       0          |  0.4998
  (0,4)     |       0          |  0.4997
  (0,5)     |       0          |  0.5000
  (1,2)     |       1          |  0.5007
  (1,3)     |       0          |  0.4990
  (1,4)     |       0          |  0.4992
  (1,5)     |       0          |  0.4996
  (2,3)     |       1          |  0.4972
  (2,4)     |       0          |  0.4975
  (2,5)     |       0          |  0.4990
  (3,4)     |       1          |  0.5031
  (3,5)     |       1          |  0.5013
  (4,5)     |       1          |  0.5011


In [2]:
# Simple gradient step to push adjacent pairs closer
eta = 0.1
epochs = 200
losses = []
for epoch in range(epochs):
    total_loss = 0.0
    for u in range(n_nodes):
        for v in range(u+1, n_nodes):
            y = 1.0 if v in adj[u] else 0.0
            pred = sigmoid(dot(Z[u], Z[v]))
            err = pred - y
            total_loss += err**2
            for l in range(d):
                grad = 2*err*pred*(1-pred)
                Z[u][l] -= eta * grad * Z[v][l]
                Z[v][l] -= eta * grad * Z[u][l]
    losses.append(total_loss)

print(f"Loss: epoch 1={losses[0]:.4f}, epoch 100={losses[99]:.4f}, epoch 200={losses[199]:.4f}")

# Check if adjacent nodes now have higher similarity
print("""
After training:""")
print("Adjacent pairs (should be high):")
for u,v in edges[:4]:
    print(f"  ({u},{v}): sim = {sigmoid(dot(Z[u],Z[v])):.4f}")
print("""Non-adjacent pairs (should be low):""")
for u,v in [(0,3),(0,4),(1,5)]:
    print(f"  ({u},{v}): sim = {sigmoid(dot(Z[u],Z[v])):.4f}")


Loss: epoch 1=3.7392, epoch 100=0.6389, epoch 200=0.0923

After training:
Adjacent pairs (should be high):
  (0,1): sim = 0.9976
  (0,2): sim = 0.9233
  (1,2): sim = 0.9203
  (3,4): sim = 0.9242
Non-adjacent pairs (should be low):
  (0,3): sim = 0.0794
  (0,4): sim = 0.0022
  (1,5): sim = 0.0023


## Preview: Three Approaches

| Method | Encoder | Similarity $S_{uv}$ | Scale |
|--------|---------|---------------------|-------|
| **DeepWalk** | Node lookup table | Random walk co-occurrence | Transductive |
| **Node2Vec** | Node lookup table | Biased random walk co-occurrence | Transductive |
| **GCN/GraphSAGE** | Multi-layer GNN | Feature-aware neighborhood | Inductive |

**Transductive:** embeddings for training nodes only; must retrain for new nodes.

**Inductive:** embedding function generalizes to unseen nodes (important for dynamic graphs).